# Model Mayhem: PCB Defect Detection (GPU Training & Evaluation)

**Checklist before running:**
1. Click **Runtime -> Change runtime type -> T4 GPU** (or L4/A100).
2. Run **Cell 1** to verify GPU and install `ultralytics`.
3. Run **Cell 2** and upload `colab_code.zip` (~23 KB) from `/home/alokd/Documents/Model Mayhem/`.
4. Run **Cell 3** to load the dataset (paste Roboflow download URL for ~3s cloud fetch, auto-load from Google Drive, or upload the ZIP).
5. Run **Cell 4** (`YOLO11s @ 1024`, ~14 mins) — optionally run **Cell 5** (`YOLO11m @ 960`, ~18 mins) + **Cell 6** (auto-tunes TTA / Tiling / Confidence / WBF on exact `mAP@50:90`) -> **Cell 7** exports `weights` + `predictions.csv`!


In [ ]:
#@title 1. Verify NVIDIA GPU & Install Dependencies
!nvidia-smi
!pip install -q "ultralytics>=8.3.0" pyyaml pandas opencv-python-headless tqdm

import torch
assert torch.cuda.is_available(), "CUDA GPU not active! Go to Runtime -> Change runtime type -> T4 GPU."
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))


In [ ]:
#@title 2. Load Project Code (`colab_code.zip`) & Run Unit Tests
import os, zipfile
from pathlib import Path

os.makedirs("/content/model_mayhem", exist_ok=True)
%cd /content/model_mayhem

if Path("colab_code.zip").exists():
    zip_file = "colab_code.zip"
else:
    from google.colab import files
    print("Please select colab_code.zip (~23 KB) from your Model Mayhem folder:")
    uploaded = files.upload()
    zip_file = next(k for k in uploaded.keys() if k.endswith(".zip"))

with zipfile.ZipFile(zip_file, "r") as zf:
    zf.extractall(".")

!python3 -m unittest discover -s tests -v


In [ ]:
#@title 3. Download/Load Dataset & Build Leak-Free Split
#@markdown **Fastest option:** Paste your Roboflow YOLOv8 download URL below (downloads in ~3s), OR leave blank to auto-search Google Drive / upload ZIP.
ROBOFLOW_ZIP_URL = ""  #@param {type:"string"}

from pathlib import Path
import glob

%cd /content/model_mayhem
dataset_zip = None

if ROBOFLOW_ZIP_URL.strip():
    !curl -L "{ROBOFLOW_ZIP_URL.strip()}" -o dataset.zip
    dataset_zip = "dataset.zip"
elif Path("pcb defect detection.v2i.yolov8.zip").exists():
    dataset_zip = "pcb defect detection.v2i.yolov8.zip"
elif Path("dataset.zip").exists():
    dataset_zip = "dataset.zip"
else:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        hits = glob.glob("/content/drive/MyDrive/**/*pcb*yolov8*.zip", recursive=True) + glob.glob("/content/drive/MyDrive/**/pcb*.zip", recursive=True)
        if hits:
            dataset_zip = hits[0]
            print("Found dataset in Google Drive:", dataset_zip)
    except Exception as e:
        print("Drive check skipped:", e)

if dataset_zip is None:
    from google.colab import files
    print("Upload pcb defect detection.v2i.yolov8.zip:")
    uploaded = files.upload()
    dataset_zip = next(k for k in uploaded.keys() if k.endswith(".zip"))

!python3 scripts/prepare_data.py --zip "{dataset_zip}" --out data --val-ratio 0.14 --copy


In [ ]:
#@title 4. Train Upgraded Orientation-Hardened YOLO11m @ 768 (~30 mins on T4)
%cd /content/model_mayhem
from pathlib import Path
import shutil

# Preserve any existing stage2/HiRes best.pt before overwriting weights/best.pt
cur_best = Path("weights/best.pt")
hires_bak = Path("weights/hires_stage2_best.pt")
if cur_best.exists() and not hires_bak.exists():
    shutil.copy2(cur_best, hires_bak)
    print("Backed up existing HiRes specialist checkpoint ->", hires_bak)

# Clear stale .npy disk cache so resized 768px cache initializes cleanly
for npy in Path("data/clean").rglob("*.npy"):
    npy.unlink()

!python3 train.py \
  --model yolo11m.pt \
  --data data/clean/data.yaml \
  --imgsz 768 \
  --epochs 28 \
  --batch -1 \
  --workers 2 \
  --lr0 0.001 \
  --cls-gain 1.0 \
  --box-gain 7.5 \
  --degrees 15.0 \
  --shear 6.0 \
  --perspective 0.0003 \
  --mosaic 0.35 \
  --scale 0.25 \
  --close-mosaic 8 \
  --cache disk \
  --name pcb_yolo11m_768


In [ ]:
#@title 5. (Optional From-Scratch Specialist) Train YOLO11s @ 1024
#@markdown Skip if `weights/hires_stage2_best.pt` already exists from an earlier run!
%cd /content/model_mayhem
from pathlib import Path

if Path("weights/hires_stage2_best.pt").exists():
    print("Found existing HiRes specialist: weights/hires_stage2_best.pt (skipping re-train)")
else:
    !python3 train.py \
      --model yolo11s.pt \
      --data data/clean/data.yaml \
      --imgsz 1024 \
      --epochs 40 \
      --batch -1 \
      --workers 2 \
      --box-gain 8.5 \
      --cls-gain 0.75 \
      --degrees 10.0 \
      --shear 4.0 \
      --mosaic 0.40 \
      --close-mosaic 10 \
      --cache disk \
      --name pcb_yolo11s_1024 \
      --no-overwrite-best


In [ ]:
#@title 6. Systematic Inference Leaderboard & Multi-Res WBF Tuning (Clean Unseen Val)
%cd /content/model_mayhem
from pathlib import Path
from infer import fuse_prediction_csvs, run_inference
from src.constants import IMAGE_EXTENSIONS
from src.metrics import evaluate_predictions, format_evaluation_report, load_ground_truth_yolo, load_predictions_csv
from src.utils import read_image_size

val_imgs = Path("data/clean/val/images")
val_lbls = Path("data/clean/val/labels")
gt_map = load_ground_truth_yolo(val_imgs, val_lbls)

m_ckpt = Path("weights/pcb_yolo11m_768_best.pt")
if not m_ckpt.exists():
    m_ckpt = Path("weights/best.pt")

s_ckpt = None
for cand in [Path("weights/hires_stage2_best.pt"), Path("weights/pcb_yolo11s_1024_best.pt")]:
    if cand.exists() and cand.resolve() != m_ckpt.resolve():
        s_ckpt = cand
        break

single_specs = []
if m_ckpt.exists():
    single_specs.extend([
        ("11m_768_std",  m_ckpt, 768,  False, False),
        ("11m_768_tta",  m_ckpt, 768,  True,  False),
        ("11m_768_rot4", m_ckpt, 768,  False, True),
        ("11m_896_rot4", m_ckpt, 896,  False, True),
    ])
if s_ckpt is not None:
    single_specs.extend([
        ("11s_1024_std",  s_ckpt, 1024, False, False),
        ("11s_1024_tta",  s_ckpt, 1024, True,  False),
        ("11s_1024_rot4", s_ckpt, 1024, False, True),
    ])

leaderboard = []
model_outputs = {}
for tag, ckpt, sz, use_tta, use_rot in single_specs:
    out_csv = Path(f"runs/bench_{tag}.csv")
    run_inference(
        source=val_imgs,
        weights=[ckpt],
        output_csv=out_csv,
        imgsz=sz,
        conf=0.01,
        iou=0.60,
        tta=use_tta,
        rot_tta=use_rot,
        tile_threshold=0,
        validate=True,
    )
    preds = load_predictions_csv(out_csv)
    metrics = evaluate_predictions(gt_map, preds)
    spec = {"kind": "single", "ckpt": ckpt, "imgsz": sz, "tta": use_tta, "rot_tta": use_rot, "iou": 0.60, "conf": 0.01}
    leaderboard.append((metrics["mAP@50:90"], metrics["mAP@50"], metrics["macro_recall@50"], tag, out_csv, spec))
    model_outputs[tag] = (metrics["mAP@50:90"], out_csv, spec)

# Sweep Multi-Model WBF combinations on CPU (~0.2s per config) using best 11m + best 11s predictions
m_cands = sorted([(s, tag, csv_p, sp) for tag, (s, csv_p, sp) in model_outputs.items() if tag.startswith("11m")], reverse=True)
s_cands = sorted([(s, tag, csv_p, sp) for tag, (s, csv_p, sp) in model_outputs.items() if tag.startswith("11s")], reverse=True)

if m_cands and s_cands:
    _, m_tag, m_csv, m_sp = m_cands[0]
    _, s_tag, s_csv, s_sp = s_cands[0]
    for w_pair in ([1.0, 1.0], [1.5, 1.0], [2.0, 1.0], [1.0, 1.5]):
        for wiou in (0.50, 0.55, 0.60):
            ftag = f"wbf_{m_tag}+{s_tag}_w{w_pair[0]}_{w_pair[1]}_i{wiou}"
            fcsv = Path(f"runs/bench_{ftag}.csv")
            fuse_prediction_csvs(
                [m_csv, s_csv],
                fcsv,
                weights=w_pair,
                wbf_iou=wiou,
                iou=0.60,
                images_dir=val_imgs,
                validate=True,
            )
            preds = load_predictions_csv(fcsv)
            metrics = evaluate_predictions(gt_map, preds)
            fspec = {"kind": "fuse", "sources": [m_sp, s_sp], "weights": w_pair, "wbf_iou": wiou, "iou": 0.60}
            leaderboard.append((metrics["mAP@50:90"], metrics["mAP@50"], metrics["macro_recall@50"], ftag, fcsv, fspec))

leaderboard.sort(key=lambda x: (x[0], x[1], x[2]), reverse=True)

print("\n" + "=" * 92)
print(f"{'Config':<48} {'mAP@50:90':>12} {'mAP@50':>12} {'Macro Rec':>12}")
print("-" * 92)
for map5090, map50, rec, tag, *_ in leaderboard[:12]:
    print(f"{tag:<48} {map5090:>12.4f} {map50:>12.4f} {rec:>12.4f}")
print("=" * 92)

WIN_MAP5090, WIN_MAP50, WIN_REC, WIN_TAG, WIN_VAL_CSV, WIN_SPEC = leaderboard[0]
print(f"WINNER: {WIN_TAG} -> mAP@50:90={WIN_MAP5090:.4f}, mAP@50={WIN_MAP50:.4f}, MacroRec={WIN_REC:.4f}")

win_preds = load_predictions_csv(WIN_VAL_CSV)
print(format_evaluation_report(evaluate_predictions(gt_map, win_preds)))

# Resolution breakdown on unseen clean/val
imgs_640 = {p.name for p in val_imgs.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS and read_image_size(p) == (640, 640)}
imgs_hi  = {p.name for p in val_imgs.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS and read_image_size(p) != (640, 640)}
for lbl, sub in [("clean/val 640x640", imgs_640), ("clean/val HiRes", imgs_hi)]:
    sub_gt = {k: v for k, v in gt_map.items() if k in sub}
    sub_p  = [r for r in win_preds if r[0] in sub]
    r_sub  = evaluate_predictions(sub_gt, sub_p)
    print(f"  {lbl:<20} ({len(sub_gt):3d} imgs): mAP@50:90={r_sub['mAP@50:90']:.4f} | mAP@50={r_sub['mAP@50']:.4f} | Rec={r_sub['macro_recall@50']:.4f}")


In [ ]:
#@title 7. Generate Final Predictions CSV with Winning Config & Download Bundle
%cd /content/model_mayhem
from pathlib import Path
import zipfile
from infer import draw_and_save_visual, fuse_prediction_csvs, run_inference
from src.constants import IMAGE_EXTENSIONS
from src.metrics import evaluate_predictions, format_evaluation_report, load_ground_truth_yolo, load_predictions_csv
from validate_submission import validate_submission_csv

test_dir = Path("data/raw/test/images")
test_lbl = Path("data/raw/test/labels")

if "WIN_SPEC" in globals():
    spec = WIN_SPEC
    print(f"Applying Cell 6 winner: {WIN_TAG}")
else:
    spec = {"kind": "single", "ckpt": Path("weights/best.pt"), "imgsz": 768, "tta": False, "rot_tta": True, "iou": 0.60, "conf": 0.01}

if spec["kind"] == "single":
    run_inference(
        source=test_dir,
        weights=[spec["ckpt"]],
        output_csv=Path("predictions.csv"),
        imgsz=spec["imgsz"],
        conf=spec["conf"],
        iou=spec["iou"],
        tta=spec["tta"],
        rot_tta=spec["rot_tta"],
        tile_threshold=0,
        save_vis=Path("vis_samples"),
        validate=True,
    )
else:
    part_csvs = []
    for idx, src_sp in enumerate(spec["sources"]):
        p_csv = Path(f"runs/final_part_{idx}.csv")
        run_inference(
            source=test_dir,
            weights=[src_sp["ckpt"]],
            output_csv=p_csv,
            imgsz=src_sp["imgsz"],
            conf=src_sp["conf"],
            iou=src_sp["iou"],
            tta=src_sp["tta"],
            rot_tta=src_sp["rot_tta"],
            tile_threshold=0,
            validate=False,
        )
        part_csvs.append(p_csv)
    fuse_prediction_csvs(
        part_csvs,
        Path("predictions.csv"),
        weights=spec["weights"],
        wbf_iou=spec["wbf_iou"],
        iou=spec["iou"],
        images_dir=test_dir,
        validate=True,
    )
    by_img = {}
    for img_n, cid, sc, x1, y1, x2, y2 in load_predictions_csv(Path("predictions.csv")):
        by_img.setdefault(img_n, []).append((cid, sc, x1, y1, x2, y2))
    for p in [x for x in sorted(test_dir.iterdir()) if x.suffix.lower() in IMAGE_EXTENSIONS][:12]:
        if p.name in by_img:
            draw_and_save_visual(p, by_img[p.name], Path("vis_samples") / p.name)

ok, errs, stats = validate_submission_csv(Path("predictions.csv"), images_dir=test_dir)
print("Audit stats:", stats)
assert ok, errs

val_report_txt = ""
if "WIN_VAL_CSV" in globals():
    gt_val = load_ground_truth_yolo("data/clean/val/images", "data/clean/val/labels")
    val_report_txt = "CLEAN UNSEEN VAL REPORT:\n" + format_evaluation_report(evaluate_predictions(gt_val, load_predictions_csv(WIN_VAL_CSV))) + "\n\n"
test_report_txt = "RAW TEST REPORT (189 imgs):\n" + format_evaluation_report(evaluate_predictions(load_ground_truth_yolo(test_dir, test_lbl), load_predictions_csv("predictions.csv")))
Path("eval_report.txt").write_text(val_report_txt + test_report_txt, encoding="utf-8")
print(Path("eval_report.txt").read_text(encoding="utf-8"))
!head -n 10 predictions.csv

bundle_path = Path("ModelMayhem_Artifacts.zip")
with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in list(Path("weights").glob("*.pt")) + [Path("predictions.csv"), Path("eval_report.txt")]:
        if p.exists():
            zf.write(p, p.as_posix())
    for p in sorted(Path("vis_samples").glob("*.jpg"))[:12]:
        zf.write(p, p.as_posix())
    for p in Path("runs").rglob("*.png"):
        zf.write(p, p.as_posix())

print("Created bundle:", bundle_path, f"({bundle_path.stat().st_size / 1e6:.2f} MB)")
try:
    from google.colab import files
    files.download(str(bundle_path))
except Exception:
    pass
